## Data ingestion to Silver layer

### Importing dependencies

In [0]:
import pyspark.sql.functions as sf

### Reading bronze data

In [0]:
df = spark.read.table("aml_transactions.aml_transactions_schema.bronze_aml_transactions")

### Column "timestamp" format updated from string to date

In [0]:
df = df.withColumn("timestamp", sf.to_timestamp("timestamp", "yyyy/MM/dd HH:mm"))

### Columns "amount_received" and "amount_paid" format
### updated from double to decimal

In [0]:
df = df.withColumn("amount_received", sf.col("amount_received").cast("decimal(22,6)"))
df = df.withColumn("amount_paid", sf.col("amount_paid").cast("decimal(22,6)"))

### Dropping exact duplicates

In [0]:
df = df.dropDuplicates()

### Validations

In [0]:
%skip
df.select("is_laundering").distinct().show()
df.agg(sf.max("amount_received")).show()
df.agg(sf.min("amount_received")).show()
df.agg(sf.max("amount_paid")).show()
df.agg(sf.min("amount_paid")).show()

In [0]:
%skip
print(df.filter(sf.col("timestamp").isNull()).limit(1).count())
df.agg(sf.max("timestamp")).show()
df.agg(sf.min("timestamp")).show()

In [0]:
%skip
print(df.filter(sf.col("timestamp").isNull()).limit(1).count())
print(df.filter(sf.col("from_bank").isNull()).limit(1).count())
print(df.filter(sf.col("from_account").isNull()).limit(1).count())
print(df.filter(sf.col("to_bank").isNull()).limit(1).count())
print(df.filter(sf.col("to_account").isNull()).limit(1).count())
print(df.filter(sf.col("amount_received").isNull()).limit(1).count())
print(df.filter(sf.col("receiving_currency").isNull()).limit(1).count())
print(df.filter(sf.col("amount_paid").isNull()).limit(1).count())
print(df.filter(sf.col("payment_currency").isNull()).limit(1).count())
print(df.filter(sf.col("payment_format").isNull()).limit(1).count())
print(df.filter(sf.col("is_laundering").isNull()).limit(1).count())

In [0]:
%skip
print(df.filter(sf.col("timestamp").rlike(r"^\s*$")).count())
print(df.filter(sf.col("from_bank").rlike(r"^\s*$")).count())
print(df.filter(sf.col("from_account").rlike(r"^\s*$")).count())
print(df.filter(sf.col("to_bank").rlike(r"^\s*$")).count())
print(df.filter(sf.col("to_account").rlike(r"^\s*$")).count())
print(df.filter(sf.col("amount_received").rlike(r"^\s*$")).count())
print(df.filter(sf.col("receiving_currency").rlike(r"^\s*$")).count())
print(df.filter(sf.col("amount_paid").rlike(r"^\s*$")).count())
print(df.filter(sf.col("payment_currency").rlike(r"^\s*$")).count())
print(df.filter(sf.col("payment_format").rlike(r"^\s*$")).count())
print(df.filter(sf.col("is_laundering").rlike(r"^\s*$")).count())

In [0]:
%skip
df.select("receiving_currency").distinct().show()
df.select("payment_currency").distinct().show()
df.select("payment_format").distinct().show()

In [0]:
%skip
df.count()

In [0]:
%skip
df.printSchema()
print(df.count())

In [0]:
%skip
print(df.filter(sf.col("amount_received") <= 0).count())
print(df.filter(sf.col("amount_received") > 0).count())
print(df.agg(sf.max("amount_received")).first()[0])
print(df.agg(sf.min("amount_received")).first()[0])
print()
print(df.filter(sf.col("amount_paid") <= 0).count())
print(df.filter(sf.col("amount_paid") > 0).count())
print(df.agg(sf.max("amount_paid")).first()[0])
print(df.agg(sf.min("amount_paid")).first()[0])

In [0]:
%skip
print(df.filter(sf.col("is_laundering") == 0).count())
print(df.filter(sf.col("is_laundering") == 1).count())

### Ingestion to Silver layer

In [0]:
silver_aml_transactions_path = "aml_transactions.aml_transactions_schema.silver_aml_transactions"

df.write.format("delta").mode("overwrite").saveAsTable(silver_aml_transactions_path)

### Post ingestion validations

In [0]:
%sql
select count(*) from aml_transactions.aml_transactions_schema.silver_aml_transactions;

count(*)
5078336


In [0]:
%sql
describe aml_transactions.aml_transactions_schema.silver_aml_transactions;

col_name,data_type,comment
timestamp,timestamp,null
from_bank,int,null
from_account,string,null
to_bank,int,null
to_account,string,null
amount_received,"decimal(22,6)",null
receiving_currency,string,null
amount_paid,"decimal(22,6)",null
payment_currency,string,null
payment_format,string,null


In [0]:
%sql
select count(*) from aml_transactions.aml_transactions_schema.silver_aml_transactions
where is_laundering = 0;

count(*)
5073159


In [0]:
%sql
select count(*) from aml_transactions.aml_transactions_schema.silver_aml_transactions
where is_laundering = 1;

count(*)
5177
